# Uplift Modeling

Who is most likely to subscribe *because we contact them*?

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/processed/model_data.csv")

# Create an observational treatment variable
df["treated"] = (df["campaign"] > 0).astype(int)
df["treated"].value_counts()

In [ ]:
treatment_summary = (
    df.groupby("treated")["y"]
      .agg(
          customers="count",
          conversions="sum",
          conversion_rate="mean"
      )
)

treatment_summary["conversion_rate"] *= 100
treatment_summary

In [ ]:
treated_rate = df.loc[df["treated"] == 1, "y"].mean()
untreated_rate = df.loc[df["treated"] == 0, "y"].mean()
naive_uplift = treated_rate - untreated_rate

print(f"Treated conversion: {treated_rate:.2%}")
print(f"Untreated conversion: {untreated_rate:.2%}")
print(f"Naive uplift: {naive_uplift:.2%}")

In [ ]:
model_df = df.copy()

model_df["treated_age"] = model_df["treated"] * model_df["age"]
model_df["treated_balance"] = model_df["treated"] * model_df["balance"]
model_df["treated_campaign"] = model_df["treated"] * model_df["campaign"]

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

propensity_features = [
    "age", "job", "marital", "education", "default", "balance",
    "housing", "loan", "day", "month", "pdays", "previous", "poutcome"
]

X_propensity = df[propensity_features]
y_treatment = df["treated"]

cat_features = X_propensity.select_dtypes(include=["object"]).columns.tolist()
num_features = X_propensity.select_dtypes(include=["int64", "float64"]).columns.tolist()

propensity_preprocessor = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), num_features),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_features)
])

propensity_model = Pipeline([
    ("preprocessor", propensity_preprocessor),
    ("model", LogisticRegression(max_iter=2000))
])

propensity_model.fit(X_propensity, y_treatment)

df["propensity_score"] = propensity_model.predict_proba(X_propensity)[:, 1]
df["propensity_score"].describe()

In [ ]:
treated = df[df["treated"] == 1].copy()
control = df[df["treated"] == 0].copy()

from xgboost import XGBClassifier

# Customer features from our original ML pipeline
features = [
    "age", "job", "marital", "education", "default", "balance",
    "housing", "loan", "day", "month", "campaign", "pdays",
    "previous", "poutcome", "age_group", "balance_group",
    "has_previous_contact", "high_campaign_contact"
]

categorical_features = df[features].select_dtypes(include=["object", "category"]).columns.tolist()
numerical_features = df[features].select_dtypes(include=["int64", "float64"]).columns.tolist()

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), numerical_features),
    ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore"))]), categorical_features)
])

model_t = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42))
])

model_c = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42))
])

model_t.fit(treated[features], treated["y"])
model_c.fit(control[features], control["y"])

df["probability_if_treated"] = model_t.predict_proba(df[features])[:, 1]
df["probability_if_control"] = model_c.predict_proba(df[features])[:, 1]

df["uplift"] = df["probability_if_treated"] - df["probability_if_control"]
df["uplift"].describe()

In [ ]:
from src.decision_engine.uplift_optimizer import calculate_incremental_value, rank_by_incremental_value

ranked_by_uplift = rank_by_incremental_value(
    df,
    uplift_column="uplift",
    conversion_value=1000,
    contact_cost=20
)

ranked_by_uplift[["uplift", "incremental_value"]].head()